In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [2]:
CSV_PATH = "raw_wholesale_customers.csv"
OUT_PATH = "segmented_wholesale_customers.csv"

In [3]:
RANDOM_STATE = 42

1-Load Dataset

In [4]:
df = pd.read_csv(CSV_PATH)
print("\n=== INITIAL SNAPSHOT ===")
print(df.head())


=== INITIAL SNAPSHOT ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


2-Select Features + IQR Cap

In [5]:
FEATURES = ["Fresh", "Milk", "Grocery",
            "Frozen", "Detergents_Paper", "Delicassen"]

In [6]:
X = df[FEATURES].copy()

In [7]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

In [8]:
low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])

In [9]:
X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

In [10]:
df[FEATURES] = X

In [11]:
print("\n=== IQR CAP SUMMARY FOR EACH FEATURE ===")
print(
    f"Fresh             low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(
    f"Milk              low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print(
    f"Grocery           low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(
    f"Frozen            low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")

print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


=== IQR CAP SUMMARY FOR EACH FEATURE ===
Fresh             low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk              low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery           low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen            low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


3-Scale Features

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)


Scaled shape: (440, 6)


4-Elbow Method

In [13]:
print("\n=== ELBOW METHOD (SSE per k) ===")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    km.fit(X_scaled)
    print(f"k={k} → SSE={km.inertia_:.2f}")


=== ELBOW METHOD (SSE per k) ===
k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


5-Train K-Means (Reproduce Lesson)

In [14]:

K = 5
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)
print("\n=== SAMPLE WITH CLUSTERS ===")
print(df.head())



=== SAMPLE WITH CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        0  
3     1788.00        3  
4     3938.25        3  


6-Evaluate K-Means

In [15]:
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
print("\n=== METRICS ===")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== METRICS ===
Silhouette Score : 0.283 (closer to +1 is better)
Davies–Bouldin   : 1.270 (lower is better)


7-Research & Train a Second Clustering Algorithm

## Second Clustering Algorithm: Agglomerative Clustering

For the second clustering algorithm, I chose Agglomerative Clustering.

Agglomerative Clustering is a hierarchical clustering method that starts by treating each customer as its own cluster. It then repeatedly merges the two most similar clusters until the desired number of clusters is reached.

This algorithm is suitable for wholesale customer segmentation because it groups customers based on their spending similarities without relying on randomly chosen cluster centers like K-Means. It can reveal hierarchical relationships between customer groups.

### Research Source

Scikit-learn Documentation:
https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html

In [16]:
from sklearn.cluster import AgglomerativeClustering

In [17]:
agg = AgglomerativeClustering(n_clusters=5)

agg_labels = agg.fit_predict(X_scaled)

In [18]:
df["Agglomerative_Cluster"] = agg_labels

In [19]:
sil = silhouette_score(X_scaled, agg_labels)
dbi = davies_bouldin_score(X_scaled, agg_labels)

print("\n=== Agglomerative Clustering Metrics ===")
print(f"Silhouette Score : {sil:.3f}")
print(f"Davies-Bouldin Index : {dbi:.3f}")


=== Agglomerative Clustering Metrics ===
Silhouette Score : 0.218
Davies-Bouldin Index : 1.325


In [20]:
print("\nCluster Counts")
print(df["Agglomerative_Cluster"].value_counts().sort_index())


Cluster Counts
Agglomerative_Cluster
0     70
1     72
2    164
3     55
4     79
Name: count, dtype: int64


In [21]:
cluster_summary = (
    df.groupby("Agglomerative_Cluster")[FEATURES]
      .mean()
      .round(2)
)

print(cluster_summary)

                          Fresh      Milk   Grocery   Frozen  \
Agglomerative_Cluster                                          
0                      22762.21   7209.70   8624.67  3708.74   
1                       5958.58  11233.14  18783.54  1435.65   
2                      11803.23   1841.65   2540.45  1671.35   
3                      11061.76   2801.73   3220.84  6864.64   
4                       5453.54   5718.85   8026.38  1120.04   

                       Detergents_Paper  Delicassen  
Agglomerative_Cluster                                
0                               2006.38     2769.90  
1                               7892.49     1392.01  
2                                411.53      777.27  
3                                567.33     1171.49  
4                               3105.71      902.95  


8-Compare Methods

In [22]:
kmeans_sil = silhouette_score(X_scaled, labels)
agg_sil = silhouette_score(X_scaled, agg_labels)
print("=== Silhouette Score Comparison ===")
print(f"K-Means                 : {kmeans_sil:.3f}")
print(f"Agglomerative Clustering: {agg_sil:.3f}")

=== Silhouette Score Comparison ===
K-Means                 : 0.283
Agglomerative Clustering: 0.218


## Comparison of Clustering Methods

The Silhouette Score was used to compare the clustering quality of the two algorithms.

* **K-Means Silhouette Score:** 0.283
* **Agglomerative Clustering Silhouette Score:** 0.218

Based on the results, **K-Means performed better** because it achieved a higher Silhouette Score than Agglomerative Clustering. A higher Silhouette Score indicates that the clusters are more compact and better separated from each other. Therefore, K-Means produced better customer segments for this wholesale customers dataset.


9-Sanity Check

In [23]:
sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]
print("\n=== SANITY CHECK (3 Clients) ===")
print(sanity)


=== SANITY CHECK (3 Clients) ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        0  


10-Save Output

In [24]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved clustered dataset → {OUT_PATH}")


Saved clustered dataset → segmented_wholesale_customers.csv
